# 03c. Переразметка

Параметры заданы в коде. Для установки зависимостей выполните в корне репозитория `python -m pip install -e ".[crowd,notebook]"`. Токен заказчика задайте в переменной `YANDEX_TASKS_OAUTH_TOKEN` или в файле `.env` в корне проекта.

Ноутбук создаёт пул и загружает задания из локального файла. Пул открывается отдельно после подготовки заданий.

Сначала выполните `03a_create_annotation.ipynb`. Используются проект и экзаменационный навык из `annotation_ids.json`.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

import requests
from dotenv import load_dotenv

start = Path.cwd().resolve()
for ROOT in (start, *start.parents):
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "butterfly_bench").is_dir():
        break
else:
    raise RuntimeError("Запустите ноутбук из папки проекта или одной из её подпапок.")
load_dotenv(ROOT / ".env")
api = requests.Session()
api.headers['Authorization'] = f"OAuth {os.environ['YANDEX_TASKS_OAUTH_TOKEN']}"
BASE_URL = 'https://tasks.yandex.ru/api/v1'


def create(path, payload, params=None):
    response = api.post(f'{BASE_URL}/{path}', json=payload, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


In [ ]:
annotation_ids = json.loads((ROOT / 'annotation_ids.json').read_text(encoding='utf-8'))
project_id = annotation_ids['project_id']
exam_skill_id = annotation_ids['exam_skill_id']

## Переразметка 43 фотографий

Параметры: 2 ₽ за страницу, перекрытие 1, лимит 15 минут.

In [ ]:
correction_pool_payload = {'project_id': project_id,
 'private_name': 'BBox — переразметка 43 фото v1',
 'public_description': 'Выделите взрослых бабочек рамками. Моль и других насекомых не выделяйте. '
                       'До 4 фото на странице.',
 'may_contain_adult_content': False,
 'reward_per_assignment': 2.0,
 'assignment_max_duration_seconds': 900,
 'auto_accept_solutions': False,
 'auto_accept_period_day': 14,
 'assignments_issuing_config': {'issue_task_suites_in_creation_order': False},
 'filter': {'and': [{'or': [{'category': 'computed',
                             'key': 'client_type',
                             'operator': 'EQ',
                             'value': 'BROWSER'}]},
                    {'or': [{'category': 'profile',
                             'key': 'languages',
                             'operator': 'IN',
                             'value': 'RU'}]},
                    {'category': 'skill',
                     'key': exam_skill_id,
                     'operator': 'GTE',
                     'value': '80'}]},
 'defaults': {'default_overlap_for_new_task_suites': 1},
 'will_expire': (datetime.now(timezone.utc) + timedelta(days=30)).strftime('%Y-%m-%dT%H:%M:%SZ')}

In [ ]:
correction_pool = create('pools', correction_pool_payload)
correction_pool_id = correction_pool['id']
correction_pool_id

### Загрузка заданий

43 фотографии для исправления разметки, 11 страниц: 10 по 4 задания и последняя из 3. Перекрытие 1.

In [ ]:
pages = json.loads((ROOT / 'crowd/annotation/tasks/reannotation.json').read_text(encoding='utf-8'))
for page in pages:
    create('task-suites', {'pool_id': correction_pool_id, **page}, params={'open_pool': 'false'})

print(f'Загружено страниц: {len(pages)}; заданий: {sum(len(p["tasks"]) for p in pages)}')

## Созданный ID

In [ ]:
reannotation_ids = {
    'project_id': project_id,
    'pool_id': correction_pool_id,
}
(ROOT / 'reannotation_ids.json').write_text(
    json.dumps(reannotation_ids, ensure_ascii=False, indent=2), encoding='utf-8'
)
reannotation_ids